# 🥚 Pelatihan YOLOv8 - Deteksi Cacat Cangkang Telur (Cracked & Dirt)
Dataset telah dikonfigurasi untuk mendeteksi 2 kelas cacat: `cracked` (retak) dan `dirt` (kotoran).
Telur utuh mulus (`good`) berfungsi sebagai **negative sample (background)** tanpa bounding box.

In [ ]:
# 1. Cek GPU Akselerasi dan Install Ultralytics YOLOv8
!nvidia-smi
!pip install -q ultralytics

In [ ]:
# 2. Ekstrak Dataset (Unggah file Egg_Dataset_Clean.zip ke Colab)
import os
import zipfile

zip_path = '/content/Egg_Dataset_Clean.zip'
extract_dir = '/content/dataset'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print(f'✅ Dataset berhasil diekstrak ke {extract_dir}!')
else:
    print('⚠️ File Egg_Dataset_Clean.zip belum diunggah. Silakan upload via panel Files (ikon folder di sebelah kiri).')

In [ ]:
# 3. Konfigurasi data.yaml untuk Google Colab (2 Kelas Cacat)
import yaml

data_config = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['cracked', 'dirt']
}

yaml_path = '/content/dataset/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print('✅ File data.yaml berhasil dikonfigurasi:')
with open(yaml_path) as f:
    print(f.read())

In [ ]:
# 4. Mulai Pelatihan Model YOLOv8 Nano
from ultralytics import YOLO

# Muat model pre-trained yolov8n.pt
model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/dataset/data.yaml',
    epochs=50,          # Jumlah epoch (rekomendasi: 50 - 100)
    imgsz=640,
    batch=16,
    device=0,           # Menggunakan GPU Colab
    project='/content/runs',
    name='egg_defect_yolov8',
    exist_ok=True,
    patience=15,
    save=True,
    verbose=True
)

In [ ]:
# 5. Evaluasi Performa pada Test Set Independen
metrics = model.val(
    data='/content/dataset/data.yaml',
    split='test',
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='eval_test'
)
print(f'Hasil Evaluasi Test: mAP50 = {metrics.box.map50:.4f}, mAP50-95 = {metrics.box.map:.4f}')

In [ ]:
# 6. Tampilkan Visualisasi Hasil Training
from IPython.display import Image, display

chart_paths = [
    '/content/runs/egg_defect_yolov8/results.png',
    '/content/runs/egg_defect_yolov8/confusion_matrix.png',
    '/content/runs/egg_defect_yolov8/val_batch0_pred.jpg'
]

for p in chart_paths:
    if os.path.exists(p):
        print(f'=== {os.path.basename(p)} ===')
        display(Image(p, width=700))


In [ ]:
# 7. Unduh Bobot Model Terbaik (best.pt) ke Komputer Anda
from google.colab import files

best_model = '/content/runs/egg_defect_yolov8/weights/best.pt'
if os.path.exists(best_model):
    print('Mengunduh best.pt...')
    files.download(best_model)
else:
    print('File best.pt belum ditemukan. Pastikan proses training telah selesai.')